# IAU Projekt - fáza 3
**Marek Kujan Privrel** a **Martin Benko** (rozdelenie práce 50:50)  
Cvičenie: Utorok 16:00 s Ing. Laurou Krajčovičovou  

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats
import statsmodels.api as sm
import statsmodels.stats as sm_stats
import sklearn
import category_encoders as ce

import math

In [2]:
train_df = pd.read_csv("train_preprocessed.csv")
test_df = pd.read_csv("test_preprocessed.csv")


In [13]:
def define_ranges(unique_values) -> dict:
    unique_values = pd.Series(unique_values).sort_values(ascending=True)
    cuts = (unique_values + unique_values.shift(1))/2
    cuts = cuts.to_list()[1:]
    
    output = {}
    for n in range(len(cuts)+1):
        output[n] = {
            "low_boundry": -np.inf if n == 0 else cuts[n-1],
            "high_boundry": np.inf if n == len(cuts) else cuts[n],
            "child_node": "-"
        }

    return output

def calc_entropy(data: pd.DataFrame): # vypočíta entropiu pre data
        p = data["failure"].value_counts(normalize=True)
        return abs((p * np.log2(p)).sum())

class Node():
    def __init__(self, data: pd.DataFrame, attribute_types: dict):
        self.attribute_types = attribute_types
        self.final = False
        self.data = data
        self.entropy = calc_entropy(self.data)
        self.children = {}   # 0: {"low_boundry": , "high_boundry": , "child_node": Node()}
        self.attribute = None
        self.prediction = None


    def pick_best_attribute(self) -> tuple:
        cols = self.data.columns.to_list()
        cols.pop(cols.index("failure"))

        best_ig = 0
        best_attr = None
        best_split_point = None

        for col in cols:
            ig, split_point = self.try_attribute(col)
            if ig > best_ig:
                best_ig = ig
                best_attr = col
                best_split_point = split_point

        return best_attr, best_split_point

    def live_and_make_children(self):
        self.prediction = round(self.data["failure"].mean())

        if self.entropy == 0:
            self.final = True
            return

        self.attribute, split_point = self.pick_best_attribute()
        if self.attribute == None:
            self.final = True
            return

        if split_point == None:   # kategorický atribút
            self.children = define_ranges(self.data[self.attribute].unique())

        else:                     # numerický atribút
            self.children = {
                0: { "low_boundry": -np.inf, "high_boundry": split_point, "child_node": "-"},
                1: { "low_boundry": split_point, "high_boundry": np.inf, "child_node": "-"}
            }
            
        for val in self.children.values():
            column = self.data[self.attribute]
            
            child_data = self.data.loc[(column > val["low_boundry"]) & (column <= val["high_boundry"])]

            if split_point is None:     # odstraňujem len kategorické atribúty, numerické ešte môžu poslúžiť
                child_data = child_data.drop(columns=self.attribute)

            val["child_node"] = Node(child_data, self.attribute_types)
            val["child_node"].live_and_make_children()

            
    def try_attribute(self, attribute: str) -> tuple:

        if self.attribute_types[attribute] == "categorical":      # Kategorické atribúty == nunique pod 100 hodnôt spracujeme podľa ID3 algoritmu

            categories = pd.DataFrame(self.data[attribute].value_counts(normalize=True))

            for category in categories.index.to_list():
                categories.loc[category, "entropy"] = calc_entropy(self.data.loc[self.data[attribute] == category])

            categories["product"] = categories.entropy * categories.proportion
            
            weighted_entropy = abs(np.sum(categories["product"]))
            info_gain = self.entropy - weighted_entropy
            return info_gain, None

            
        
        # Spojité atribúty - hľadáme najlepší split point, podľa ktorého bude info_gain najlepší

        temp = self.data[[attribute, "failure"]].sort_values(by=attribute)
        # temp["failure_diff"] = temp["failure"].diff()
        temp["middle"] = (temp[attribute] + temp[attribute].shift(1)) / 2

        thresholds = temp.loc[temp[attribute] != temp[attribute].shift(1), "middle"].dropna()

        thresholds = pd.DataFrame({"threshold": thresholds})

        if thresholds.empty:
            return 0, None

        
        thresholds["percentage_left"] = thresholds["threshold"].apply(lambda t: (temp[attribute] <= t).mean())
        thresholds["percentage_right"] = 1 - thresholds["percentage_left"]

        thresholds["entropy_left"] = thresholds["threshold"].apply(lambda t: calc_entropy(temp[temp[attribute] <= t]))
        thresholds["entropy_right"] = thresholds["threshold"].apply(lambda t: calc_entropy(temp[temp[attribute] > t]))

        thresholds["weighted_entropy"] = (thresholds["percentage_left"] * thresholds["entropy_left"] + thresholds["percentage_right"] * thresholds["entropy_right"])

        best_index = thresholds["weighted_entropy"].idxmin()
        info_gain = self.entropy - thresholds.at[best_index, "weighted_entropy"]
        best_split_point = thresholds.at[best_index, "threshold"]

        return info_gain, best_split_point

    def find_range(self, value):
        for key, val in self.children.items():
            if value <= val["high_boundry"]:
                if value > val["low_boundry"]:
                    return key

        raise Exception("Function 'find_range()' somehow could not find the range!")
    


In [14]:
class ID3_Classifier():
    def __init__(self):
        self.root = None

    def train(self, data):
        attribute_types = {}
        for attr in data.columns.to_list():
            attribute_types[attr] = "categorical" if data[attr].nunique() < 100 else "numeric"
        
        self.root = Node(data, attribute_types)
        self.root.live_and_make_children()

    def predict(self, test_vector):
        crossroad = self.root
        while not crossroad.final:
            index = crossroad.find_range(test_vector[crossroad.attribute])
            crossroad = crossroad.children[index]["child_node"]

        return crossroad.prediction

id3 = ID3_Classifier()
id3.train(train_df)

vector = train_df.iloc[10]
print(id3.predict(vector))

1


In [15]:
vector = train_df.iloc[5]
print(id3.predict(vector))

0
